In [ ]:
import duckdb
from   deltalake import DeltaTable, write_deltalake
import pyarrow as pa
con = duckdb.connect()
TABLE = "abfss://tpch@onelake.dfs.fabric.microsoft.com/storage.Lakehouse/Tables/T10/sail"
KEY = ["UNIT", "SETTLEMENTDATE", "INTERVENTION","YEAR"]
dt = DeltaTable(TABLE)
current_version = dt.version()
con.sql(f""" ATTACH '{TABLE}' AS scada (TYPE delta, VERSION {current_version}); """)
result = con.sql(F""" SELECT *  FROM scada limit 10 """).arrow()
target_schema = pa.schema(dt.schema().to_arrow())
result = result.cast(target_schema)
predicate = " AND ".join(f"t.{k} = s.{k}" for k in KEY)

dt.update_incremental()
if dt.version() != current_version:
    raise ValueError(f"Snapshot isolation violation: table version changed from {current_version} to {dt.version()}")

(
    dt.merge(
        source=result,
        predicate=predicate,
        source_alias="s",
        target_alias="t",
    )
    .when_not_matched_insert_all()
    .execute()
)